## Setup

In [17]:
%load_ext autoreload
%autoreload 0

from ble import get_ble_controller
from base_ble import LOG
from cmd_types import CMD
import time
import numpy as np
import threading
import asyncio

LOG.propagate = False

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Connect

In [23]:
# Get ArtemisBLEController object
ble = get_ble_controller()

# Connect to the Artemis Device
ble.reload_config()
ble.connect()

2026-01-25 18:53:09,279 | INFO     |: Looking for Artemis Nano Peripheral Device: c0:42:d1:79:ab:49
2026-01-25 18:53:09,279 | INFO     |: Scanning for device with address: c0:42:d1:79:ab:49, service UUID: 4843fd62-068c-4a37-af34-e724c6a05681
2026-01-25 18:53:19,399 | INFO     |: Found 24 total devices
2026-01-25 18:53:19,399 | INFO     |: Found matching device: C0:42:D1:79:AB:49 (name: Artemis BLE)
2026-01-25 18:53:20,342 | INFO     |: Connected to c0:42:d1:79:ab:49


## Disconnect

In [ ]:
ble.disconnect()

## Echo

In [ ]:
# Send the message
ble.send_command(CMD.ECHO, "Hello World!")

# Check that the board received it and returns it
s = ble.receive_string(ble.uuid['RX_STRING'])
print("Recieved from board: " + s)

## Send Three Floats

In [ ]:
ble.send_command(CMD.SEND_THREE_FLOATS, "3.14|5.25|7.432")

## Get Time Millis

In [ ]:
# Send command
ble.send_command(CMD.GET_TIME_MILLIS, "")

# Check board message
s = ble.receive_string(ble.uuid['RX_STRING'])
print("Recieved from board: " + s)

## Notification Handler

In [ ]:
def notification_callback(uuid, charac_bytearray):
    msg = ble.bytearray_to_string(charac_bytearray)
    if msg.startswith("T:"):
        t_ms = int(msg.split(":")[1])
        print("Time (ms):", t_ms)
    else:
        print("Unexpected message:", msg)

duration_s = 0.3  # set collection time

# Set up callback
ble.start_notify(ble.uuid['RX_STRING'], notification_callback)

t0 = time.time()
while (time.time() - t0) < duration_s:
    time.sleep(0.01)   # small sleep so you don't busy-wait

ble.stop_notify(ble.uuid['RX_STRING'])
print("End time notify")




## Time Array

In [35]:
done = threading.Event()
time_values = []

def time_array_notification_callback(uuid, data: bytearray):
    msg = ble.bytearray_to_string(data).strip()
    print("Msg received:", repr(msg))

    parts = [p.strip() for p in msg.split(",") if p.strip()]
    if parts and parts[-1].lower() == "end":
        time_values.extend(parts[:-1])
        print("End of values")
        done.set()
        return
    time_values.extend(parts)

def get_time_data(timeout_s=10.0):
    time_values.clear()
    done.clear()
    
    # send command (NO await)
    ble.send_command(CMD.SEND_TIME_DATA, "")

    t0 = time.time()
    while not done.is_set() and (time.time() - t0) < timeout_s:
        ble.sleep(0.01)

    if not done.is_set():
        print("Timeout reached.")
    return list(time_values)

# start notify
try:
    ble.start_notify(ble.uuid['RX_STRING'], time_array_notification_callback)
except Exception as e:
    if "Notify acquired" in str(e):
        print("Notify already active; continuing.")
time.sleep(0.2)

vals = get_time_data()
print(vals)

try:
    ble.stop_notify(ble.uuid['RX_STRING'])
except Exception as e:
    print("Failed to stop notifications with exception: ", e)


Msg received: '39549,39549,39549,39549,39550,39550,39550,39554,39549,39549,end'
End of values
['39549', '39549', '39549', '39549', '39550', '39550', '39550', '39554', '39549', '39549']
